# GSoC next-year organization prediction

This notebook uses the five-year workbook history as a rolling supervised-learning problem. The training label is `T-1`; its features are `T-2` through `T-5`. After evaluation, the model is refit on all organizations and applied to `T-1` through `T-4` to predict participation in the next/current target year `T`.

The prediction is about organization presence in the GSoC list, not acceptance of a particular project.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = [Path.cwd(), Path.cwd() / 'src' / 'backend']
BACKEND = next((path for path in candidates if (path / 'gsoc_organizations.xlsx').exists()), candidates[-1])
WORKBOOK = BACKEND / 'gsoc_organizations.xlsx'
PREDICTIONS = BACKEND / 'gsoc_t_year_predictions.csv'
assert WORKBOOK.exists(), f'Workbook not found: {WORKBOOK}'


In [2]:
orgs = pd.read_excel(WORKBOOK, sheet_name='Orgs')
scrape_log = pd.read_excel(WORKBOOK, sheet_name='Scrape log')
required = ['Organization', 'T-1', 'T-2', 'T-3', 'T-4', 'T-5']
missing = sorted(set(required) - set(orgs.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

scrape_log['year'] = pd.to_numeric(scrape_log['year'], errors='coerce')
scrape_log['total_orgs'] = pd.to_numeric(scrape_log['total_orgs'], errors='coerce')
scrape_log = scrape_log.dropna(subset=['year']).astype({'year': int})
current_year = int(scrape_log['year'].max())
current_total = int(scrape_log.loc[scrape_log['year'].eq(current_year), 'total_orgs'].iloc[0])
target_year = current_year + 1 if current_total > 0 else current_year

presence_cols = ['T-1', 'T-2', 'T-3', 'T-4', 'T-5']
for col in presence_cols:
    orgs[col] = orgs[col].map({True: 1, False: 0, 1: 1, 0: 0}).astype('Int64')

print(f'Organizations: {len(orgs):,}')
print(f'Current scraped year: {current_year}; prediction target T: {target_year}')
print(scrape_log.to_string(index=False))


Organizations: 419
Current scraped year: 2026; prediction target T: 2027
 year  total_orgs
 2026         183
 2025         185
 2024         194
 2023         168
 2022         198
 2021         202
 2020         199


## Shifted training setup

For every organization, the training target is `T-1`. The four input features are the four older flags. This makes the model learn the transition from an organization's recent participation history to the next year. The same four-position feature layout is then shifted forward for the T-year prediction.

In [3]:
feature_cols = ['T-2', 'T-3', 'T-4', 'T-5']
X = orgs[feature_cols].fillna(0).astype(float)
y = orgs['T-1'].fillna(0).astype(int)

models = {
    'logistic_regression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
    'random_forest': RandomForestClassifier(n_estimators=400, max_depth=4, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1),
}
if y.nunique() < 2:
    raise ValueError('T-1 has only one class; a classifier cannot be trained.')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for name, model in models.items():
    result = cross_validate(model, X, y, cv=cv, scoring=['accuracy', 'balanced_accuracy', 'roc_auc'])
    scores.append({
        'model': name,
        'accuracy_mean': result['test_accuracy'].mean(),
        'balanced_accuracy_mean': result['test_balanced_accuracy'].mean(),
        'roc_auc_mean': result['test_roc_auc'].mean(),
    })
scores = pd.DataFrame(scores).sort_values(['balanced_accuracy_mean', 'roc_auc_mean'], ascending=False)
display(scores)


,model,accuracy_mean,balanced_accuracy_mean,roc_auc_mean
0,logistic_regression,0.809122,0.806637,0.842265
1,random_forest,0.792456,0.789793,0.855977


In [23]:
best_name = scores.iloc[0]['model']
best_model = clone(models[best_name]).fit(X, y)

# Shift the feature window forward one year: T-1 becomes the newest input.
prediction_features = ['T-1', 'T-2', 'T-3', 'T-4']
X_target = orgs[prediction_features].fillna(0).astype(float)
# Keep the feature names identical to training; only the values move forward one year.
X_target.columns = feature_cols
predicted = best_model.predict(X_target).astype(bool)
probability = best_model.predict_proba(X_target)[:, 1]

predictions = orgs[['Organization', 'Description', 'Tech', 'Topic', 'Website', 'GSoC-link']].copy()
predictions[f'Predicted {target_year}'] = predicted
predictions[f'Probability {target_year}'] = probability.round(4)
predictions['Model'] = best_name
predictions.to_csv(PREDICTIONS, index=False)

print(f'Best model: {best_name}')
print(f'Predicted organizations for T={target_year}: {int(predicted.sum()):,} of {len(predicted):,}')
display(predictions.sort_values(f'Probability {target_year}', ascending=False).head(20))
print(f'Saved predictions to {PREDICTIONS}')


Best model: logistic_regression
Predicted organizations for T=2027: 183 of 419


,Organization,Description,Tech,Topic,Website,GSoC-link,Predicted 2027,Probability 2027,Model
31,OpenMRS,"Write Code, Save Lives.","mysql, javascript, java, reactjs, fhir","platform, frontend, Electronic Medical Record ...",https://openmrs.org/,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
20,GNU Radio,The free & open software radio ecosystem,"python, c++, qt, simd","real-time, dsp, communications engineering, cy...",https://www.gnuradio.org,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
30,OpenAstronomy,Look at the Universe with the power of Open So...,"c, python, c++, julia","image processing, astronomy, data analysis, so...",https://openastronomy.org/,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
29,NumFOCUS,NumFOCUS promotes open source scientific softw...,"python, c++, r, julia","data science, graphics, ai, scientific computi...",https://numfocus.org,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
28,National Resource for Network Biology (NRNB),Developing open source tools for network biology,"python, javascript, html, css, LLM","machine learning, web application, data scienc...",https://nrnb.org,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
26,MetaBrainz Foundation Inc,"Open music / book metadata, music recommendations","python, machine learning, perl, postgres, spark","machine learning, open data, music, books, mus...",https://policies.google.com/technologies/cooki...,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
27,MIT App Inventor,Anyone can build apps with global impact,"javascript, java, gwt, swift","education, development tools, android, ios",http://appinventor.mit.edu,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
25,MDAnalysis,Analysis of molecular simulations data with Py...,"python, cython, c/c++","science, computational-chemistry, high-perform...",https://www.mdanalysis.org,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
24,Liquid Galaxy project,We code immersive and interactive apps with GE...,"linux, android, nodejs, flutter, Google Earth","visualization, artificial intelligence, networ...",https://www.liquidgalaxy.eu,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression
18,GNU Compiler Collection (GCC),GNU compilers,"c/c++, gnu make, gnu autotools","compilers, developer tools, toolchain, openmp,...",https://policies.google.com/technologies/cooki...,https://summerofcode.withgoogle.com/programs/2...,True,0.8673,logistic_regression


Saved predictions to c:\Users\hridd\Documents\VSCode\Web_Dev\Projects\Advanced\orglytics\src\backend\gsoc_t_year_predictions.csv


In [22]:
predictions["Predicted 2027"].value_counts()

Predicted 2027
False    236
True     183
Name: count, dtype: int64